In [70]:
import sys

In [71]:
def chain_matrix(p):
  n=len(p)
  m=[[0 for _ in range(n)] for _ in range(n)]
  split=[[0 for _ in range(n)] for _ in range(n)]

  for L in range(2,n):
    for i in range(1,n-L+1):
      j=i+L-1
      m[i][j]=sys.maxsize

      for k in range(i,j):
        cost=m[i][k]+m[k+1][j]+p[i-1]*p[k]*p[j]
        if cost<m[i][j]:
          m[i][j]=cost
          split[i][j]=k
  
  return split,m

p = [5,10,3,12,5,50,6]
split,table=chain_matrix(p)

for row in table[1:]:
  print(row[1:])

print(table[1][len(p)-1])

def print_optimal_parens(split,i,j):
  if(i==j):
    return chr(65+i-1)
  return (
    "("+print_optimal_parens(split,i,split[i][j])+print_optimal_parens(split,split[i][j]+1,j)+")"
  )

print(print_optimal_parens(split,1,len(p)-1))

[0, 150, 330, 405, 1655, 2010]
[0, 0, 360, 330, 2430, 1950]
[0, 0, 0, 180, 930, 1770]
[0, 0, 0, 0, 3000, 1860]
[0, 0, 0, 0, 0, 1500]
[0, 0, 0, 0, 0, 0]
2010
((AB)((CD)(EF)))


In [72]:
def tsp_path(graph):
  n=len(graph)
  dp=[[-1]*(1<<n) for _ in range(n)]
  parent=[[-1]*(1<<n) for _ in range(n)]

  def visit(city,mask):
    if mask==(1<<n)-1:
      return graph[city][0]
    if dp[city][mask]!=-1:
      return dp[city][mask]
    
    ans=float('inf')
    best_next_city=0

    for next_city in range(n):
      if (mask & (1<<next_city))==0:
        cost=graph[city][next_city]+visit(next_city,mask|(1<<next_city))
        if cost<ans:
          ans=cost
          best_next_city=next_city
    dp[city][mask]=ans
    parent[city][mask]=best_next_city
    return ans
  
  min_cost=visit(0,1)

  path=[]
  current_city=0
  current_mask=1

  while current_city!=-1:
    path.append(current_city)
    current_city=parent[current_city][current_mask]
    if current_city!=-1:
      current_mask|=(1<<current_city)

  return min_cost,path


graph = [
[0, 10, 15, 20],
[10, 0, 35, 25],
[15, 35, 0, 30],
[20, 25, 30, 0]
]

min_cost,path=tsp_path(graph)
print(min_cost)
print('->'.join(map(str,path)))


80
0->1->3->2


In [73]:
def warshall(graph):
  n=len(graph)

  for k in range(n):
    for i in range(n):
      for j in range(n):
        graph[i][j]=graph[i][j] or (graph[i][k] and graph[k][j])

  return graph

graph = [
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1],
[0, 0, 0, 0]
]
for row in warshall(graph):
  print(row)

[0, 1, 1, 1]
[0, 0, 1, 1]
[0, 0, 0, 1]
[0, 0, 0, 0]


In [74]:
INF = float('inf')

graph = [
[0, 3, INF, 7],
[8, 0, 2, INF],
[5, INF, 0, 1],
[2, INF, INF, 0]
]

def floyd_warshall(graph):
  n=len(graph)
  dist=[]
  store=[[None for _ in range(n)] for _ in range(n)]
  for row in graph:
    dist.append(row)

    for i in range(n):
      for j in range(n):
        if graph[i][j]!=float('inf') and i!=j:
          store[i][j]=j

  for k in range(n):
    for i in range(n):
      for j in range(n):
        if dist[i][j]>dist[i][k]+dist[k][j]:
          dist[i][j]=dist[i][k]+dist[k][j]
          store[i][j]=store[i][k]
  return dist,store

dist,store=floyd_warshall(graph)
for row in dist:
  print(row)

v1=2
v2=1
def construct_path(u,v,store):
  if store[u][v] is None:
    return None
  path=[u]
  while u!=v:
    u=store[u][v]
    path.append(u)
  return path

print("->".join(map(str,construct_path(v1,v2,store))))

[0, 3, 5, 6]
[5, 0, 2, 3]
[3, 6, 0, 1]
[2, 5, 7, 0]
2->3->0->1


In [81]:
def dijkstra_path(graph,start,end):
  distances={}
  visited=[]
  tree={}
  tree[start]=None

  for node in graph:
    distances[node]=float('inf')

  distances[start]=0

  while len(visited)<len(graph):
    min_node=None
    min_dist=float('inf')

    for node in graph:
      if node not in visited and distances[node]<min_dist:
        min_dist=distances[node]
        min_node=node
    visited.append(min_node)

    for neighbor,weight in graph[min_node].items():
      if neighbor not in visited:
        new_dist=distances[min_node]+weight

        if new_dist<distances[neighbor]:
          distances[neighbor]=new_dist
          tree[neighbor]=min_node
    
  path=[]
  node=end
  while node!=None:
    path.append(node)
    node=tree[node]
  path.reverse()
  return path,distances[end]
  
graph = {
'A': {'B': 4, 'C': 2},
'B': {'A': 4, 'C': 5, 'D': 10},
'C': {'A': 2, 'B': 5, 'D': 3},
'D': {'B': 10, 'C': 3}
}
path,distance=dijkstra_path(graph, 'A','D')
print("Shortest path:")
print("->".join(map(str,path)))
print("Total distance:",distance)
    

Shortest path:
A->C->D
Total distance: 5


In [76]:
graph = {
    "A": {"B": 10, "C": 15, "D": 20},
    "B": {"A": 10, "C": 12},
    "C": {"A": 15, "B": 12, "D": 18},
    "D": {"A": 20, "C": 18}
}

def prim(graph):
  visited=set()
  min_tree=[]
  min_weight=0

  visited.add(list(graph.keys())[0])
  while len(visited)<len(graph):
    min_edge=float('inf')
    next_vertex=None

    for vertex in visited:
      for adj_vertex,weight in graph[vertex].items():
        if adj_vertex not in visited:
          if weight<min_edge:
            min_edge=weight
            store=vertex
            next_vertex=adj_vertex

    min_tree.append((store,next_vertex,min_edge))
    visited.add(next_vertex)
    min_weight+=min_edge
  return min_tree,min_weight
mst, min_weight = prim(graph)
print("Minimum Spanning Tree: ", mst)
print("Minimum Weight: ", min_weight)


Minimum Spanning Tree:  [('A', 'B', 10), ('B', 'C', 12), ('C', 'D', 18)]
Minimum Weight:  40


In [77]:
import heapq
def generate_huffman_tree(data):
  heap=[(freq,char,None,None) for char,freq in data.items()]
  heapq.heapify(heap)

  while(len(heap)>1):
    f1,c1,l1,r1=heapq.heappop(heap)
    f2,c2,l2,r2=heapq.heappop(heap)
    merged=(f1+f2,None,(f1,c1,l1,r1),(f2,c2,l2,r2))
    heapq.heappush(heap,merged)
  return heap[0]

def traverse_tree(node,code='',huffman_codes={}):
  if node[1] is not None:
    huffman_codes[node[1]]=code
  else:
    traverse_tree(node[2],code+'0',huffman_codes)
    traverse_tree(node[3],code+'1',huffman_codes)
    return huffman_codes

data={"A": 15, "B": 9, "C": 112, "D": 13, "E": 16, "F": 45}
huffman_tree = generate_huffman_tree(data)
huffman_codes = traverse_tree(huffman_tree)
print(huffman_codes)

{'F': '00', 'B': '0100', 'D': '0101', 'A': '0110', 'E': '0111', 'C': '1'}


In [78]:
def kruskal(graph):
  parent={}
  rank={}
  def find(u):
    if u!=parent[u]:
      parent[u]=find(parent[u])
    return parent[u]
  
  def union(u,v):
    root_u=find(u)
    root_v=find(v)
    if root_u!=root_v:
      if rank[root_u]>rank[root_v]:
        parent[root_v]=root_u
      elif rank[root_u]<rank[root_v]:
        parent[root_u]=root_v
      else:
        parent[root_v]=root_u
        rank[root_u]+=1
      return True
    return False
  
  edges=[]
  for u in graph:
    parent[u]=u
    rank[u]=0
    for v,w in graph[u].items():
      if(v,u,w) not in edges:
        edges.append((u,v,w))

  edges.sort(key=lambda x:x[2])

  mst=[]
  min_weight=0

  for u,v,w in edges:
    if union(u,v):
      mst.append((u,v,w))
      min_weight+=w

  return mst,min_weight

graph = {
    "A": {"B": 10, "C": 15, "D": 20},
    "B": {"A": 10, "C": 12},
    "C": {"A": 15, "B": 12, "D": 18},
    "D": {"A": 20, "C": 18}
}
mst, weight = kruskal(graph)
print("MST:", mst)
print("Total Weight:", weight)

MST: [('A', 'B', 10), ('B', 'C', 12), ('C', 'D', 18)]
Total Weight: 40
